# Database and Business Model

This notebook sets up the database used by the AI Analytics Assistant.

The goal is to build a small but realistic e-commerce database with customers, products, orders, payments, returns and stores. I will also check the relationships between the tables and define a few business metrics that the assistant will use later.

Before connecting any AI model, I want to make sure the database itself is clear, reproducible and working correctly.

## 1. Check the project environment

Before starting the database work, I want to check that the project is using the correct Python environment and that the tools I need are available.

Python will be used for generating and validating the data, Git will track the project as it develops, and PostgreSQL will store the business data.

In [1]:
import sys
import shutil
import subprocess

# Show the Python version used by this notebook
print("Python version:", sys.version.split()[0])

# Show the exact Python environment being used
print("Python executable:", sys.executable)

# Check Git
git_version = subprocess.run(
    ["git", "--version"],
    capture_output=True,
    text=True
).stdout.strip()

print("Git:", git_version)

# Check whether PostgreSQL is available
if shutil.which("psql"):
    postgres_version = subprocess.run(
        ["psql", "--version"],
        capture_output=True,
        text=True
    ).stdout.strip()

    print("PostgreSQL:", postgres_version)
else:
    print("PostgreSQL: not found")

Python version: 3.12.11
Python executable: /Users/arvindshine/ai-analytics-assistant/.venv/bin/python
Git: git version 2.50.1 (Apple Git-155)
PostgreSQL: psql (PostgreSQL) 18.6 (Homebrew)


## 2. Start PostgreSQL and test the connection

PostgreSQL is installed, but the database server also needs to be running before I can create or query databases.

In this section, I will start the PostgreSQL service and check that I can connect to it successfully.

In [2]:
# Check that PostgreSQL is running and that we can connect to it
connection_check = subprocess.run(
    [
        "psql",
        "-d", "postgres",
        "-c", "SELECT current_database(), current_user;"
    ],
    capture_output=True,
    text=True
)

print(connection_check.stdout)

# Show any error if the connection failed
if connection_check.returncode != 0:
    print("Connection error:")
    print(connection_check.stderr)

 current_database | current_user 
------------------+--------------
 postgres         | arvindshine
(1 row)




## 3. Plan the database tables

Before creating the tables in PostgreSQL, I want to decide what information each table will store and how the tables will connect to each other.

The database will represent a small e-commerce business with customers, products, orders, payments, returns, stores and promotions. Splitting the data into related tables will make the database easier to manage and will give the assistant realistic SQL problems involving joins and aggregations.

In [3]:
# A simple overview of the tables we plan to create
table_plan = {
    "customers": {
        "stores": "Customer details such as name, region and signup date",
        "primary_key": "customer_id",
        "links_to": []
    },
    "stores": {
        "stores": "Store name, city and region",
        "primary_key": "store_id",
        "links_to": []
    },
    "categories": {
        "stores": "Product categories such as Electronics or Home",
        "primary_key": "category_id",
        "links_to": []
    },
    "products": {
        "stores": "Products, prices and category information",
        "primary_key": "product_id",
        "links_to": ["categories"]
    },
    "orders": {
        "stores": "Customer orders, dates, store and order status",
        "primary_key": "order_id",
        "links_to": ["customers", "stores"]
    },
    "order_items": {
        "stores": "Individual products included in each order",
        "primary_key": "order_item_id",
        "links_to": ["orders", "products", "promotions"]
    },
    "payments": {
        "stores": "Payments made for orders",
        "primary_key": "payment_id",
        "links_to": ["orders"]
    },
    "returns": {
        "stores": "Returned order items and refund information",
        "primary_key": "return_id",
        "links_to": ["order_items"]
    },
    "promotions": {
        "stores": "Discounts and promotional campaigns",
        "primary_key": "promotion_id",
        "links_to": []
    }
}

# Print the planned structure in a readable way
for table_name, details in table_plan.items():
    print(f"{table_name}")
    print(f"  Purpose: {details['stores']}")
    print(f"  Primary key: {details['primary_key']}")
    print(f"  Links to: {', '.join(details['links_to']) if details['links_to'] else 'None'}")
    print()

customers
  Purpose: Customer details such as name, region and signup date
  Primary key: customer_id
  Links to: None

stores
  Purpose: Store name, city and region
  Primary key: store_id
  Links to: None

categories
  Purpose: Product categories such as Electronics or Home
  Primary key: category_id
  Links to: None

products
  Purpose: Products, prices and category information
  Primary key: product_id
  Links to: categories

orders
  Purpose: Customer orders, dates, store and order status
  Primary key: order_id
  Links to: customers, stores

order_items
  Purpose: Individual products included in each order
  Primary key: order_item_id
  Links to: orders, products, promotions

payments
  Purpose: Payments made for orders
  Primary key: payment_id
  Links to: orders

returns
  Purpose: Returned order items and refund information
  Primary key: return_id
  Links to: order_items

promotions
  Purpose: Discounts and promotional campaigns
  Primary key: promotion_id
  Links to: None

